## Day 2

### Part 10: Predicting a Digit

We have a forward pass. Now let us turn the output into an actual prediction: which digit does the network think it is looking at?

The answer is simply the index of the highest probability. If the output layer gives the highest value to index 3, the network's prediction is "3".

In [ ]:
def predict(net, inputs):
    """
    Run inputs through the network and return the predicted digit.
    
    Use your forward function to get the output probabilities.
    Return the index of the maximum value in the output.
    """
    # Your code here.
    pass

In [ ]:
# Test it on a few examples from the dataset.
# At this point the predictions will almost certainly be wrong —
# the network has not learned anything yet.
for label, pixels in data[:8]:
    pred = predict(net, pixels)
    correct = '✓' if pred == label else '✗'
    print(f'  True label: {label}   Prediction: {pred}   {correct}')

Wrong predictions at this stage are expected and normal. We need a way to measure how wrong, and then a way to correct the weights.

---

### Part 11: How Wrong Are We? (Loss)

To train a network, we need a number that captures how bad a given prediction was. That number is called the **loss**.

We use **cross-entropy loss**. The idea is simple: if the network assigned probability `p` to the correct class, the loss is `-log(p)`. If `p` is close to 1 (confident and correct), the loss is close to 0. If `p` is close to 0 (confident and wrong), the loss is very large.

```
L = -log(p_correct)
```

In [ ]:
def cross_entropy_loss(output_probs, label):
    """
    Compute the cross-entropy loss for one prediction.
    
    output_probs: the softmax output, a list of 10 floats
    label:        the correct class as an integer (0-9)
    
    Returns a single float.
    
    Hint: add a tiny value (1e-12) inside the log to avoid log(0).
    """
    # Your code here.
    pass

In [ ]:
# Test it.
# If the network assigns probability 1.0 to the correct class, loss should be 0.
# If it assigns 0.1 (ten percent), loss should be higher.

perfect = [0.0] * 10
perfect[3] = 1.0
print(f'Loss when correct class gets 1.0:  {cross_entropy_loss(perfect, 3):.4f}')

uncertain = [0.1] * 10
print(f'Loss when correct class gets 0.1:  {cross_entropy_loss(uncertain, 3):.4f}')

# Try constructing a case where the network is confident but wrong.
# What is the loss?

---

### Part 12: Fixing the Weights (Backpropagation)

We know how wrong we were. Now we need to fix the weights.

**Backpropagation** is the method. The idea is to work backwards through the network, computing how much each weight contributed to the error, then nudging each weight slightly in the direction that would reduce it. The size of each nudge is controlled by the **learning rate** — a small number, typically something like 0.01.

This uses the chain rule from calculus. The cross-entropy loss and softmax combine to give a particularly clean expression for the output layer gradient:

```
delta_output[i] = output[i] - target[i]
```

where `target` is the one-hot vector for the correct class (all zeros except a 1 at the correct index). For subsequent layers we propagate this delta backwards, multiplying by the weights and by the derivative of ReLU at each step.

The derivative of ReLU is 1 where the pre-activation value was positive, and 0 otherwise.

The function below is provided. Read through it carefully. Notice:

- We save the current weights **before** updating them. Think about why this matters before moving on.
- We work backwards through the layers using `range(n_layers - 1, -1, -1)`.
- Each step computes a new `delta` for the layer below using the saved weights.

In [ ]:
# Provided: backpropagation.
# Read through this carefully before moving to Part 13.

def backprop(net, inputs, label, layer_outputs, learning_rate=0.01):
    """
    Update the weights and biases in net using one training example.
    
    net:           the network dict (modified in place)
    inputs:        the original input vector
    label:         the correct digit (integer, 0-9)
    layer_outputs: the (pre, post) pairs returned by forward()
    learning_rate: step size for weight updates
    """
    n_layers = len(net['weights'])

    # One-hot target: all zeros except a 1 at the correct class.
    target = [0.0] * 10
    target[label] = 1.0

    # Output layer gradient: cross-entropy + softmax simplify to output - target.
    output_probs = layer_outputs[-1][1]
    delta = [output_probs[i] - target[i] for i in range(len(output_probs))]

    for layer_idx in range(n_layers - 1, -1, -1):

        # The input to this layer is either the original input (layer 0)
        # or the post-activation output of the previous layer.
        if layer_idx == 0:
            layer_input = inputs
        else:
            layer_input = layer_outputs[layer_idx - 1][1]

        # Save current weights before updating.
        # We need the original weights to compute the gradient for the layer below.
        saved_weights = [row[:] for row in net['weights'][layer_idx]]

        # Update each weight and bias.
        for j in range(len(net['weights'][layer_idx])):
            for k in range(len(net['weights'][layer_idx][j])):
                # The gradient of the loss with respect to this weight is
                # delta[j] * layer_input[k].
                net['weights'][layer_idx][j][k] -= learning_rate * delta[j] * layer_input[k]
            net['biases'][layer_idx][j] -= learning_rate * delta[j]

        # Compute delta for the layer below (chain rule).
        if layer_idx > 0:
            pre_activation = layer_outputs[layer_idx - 1][0]
            new_delta = []
            for k in range(len(layer_input)):
                # Sum contributions from all nodes in the current layer.
                grad = sum(saved_weights[j][k] * delta[j]
                           for j in range(len(delta)))
                # ReLU's derivative: 1 if pre-activation was positive, else 0.
                relu_deriv = 1.0 if pre_activation[k] > 0 else 0.0
                new_delta.append(grad * relu_deriv)
            delta = new_delta

Before moving on, answer these two questions in the cell below:

1. Why do we save the weights before updating them? What would go wrong if we used the already-updated weights when computing the next layer's delta?

2. The output layer gradient simplifies to `output[i] - target[i]`. What does this expression equal when the network assigns a very high probability to the correct class?

In [ ]:
# Write your answers here as comments or a print statement.
# Q1:
# Q2:

---

### Part 13: The Training Loop

We have forward, loss, and backprop. Now we can write a training loop that puts them all together.

The loop goes through the training data one example at a time, runs a forward pass, records the loss and whether the prediction was correct, and runs backprop to update the weights. We record the loss and accuracy as we go so we can see whether the network is improving.

In [ ]:
def train(net, data, epochs=1, learning_rate=0.01):
    """
    Train the network on data for the given number of epochs.
    
    An epoch is one full pass through all the training examples.
    
    For each example:
    1. Run forward() to get layer_outputs.
    2. Record the loss using cross_entropy_loss.
    3. Record whether predict() returns the correct label.
    4. Run backprop() to update the weights.
    
    Print the average loss and accuracy after each epoch.
    
    Returns (loss_history, accuracy_history), where each is a list
    of per-example values across all epochs.
    """
    loss_history = []
    accuracy_history = []

    for epoch in range(epochs):
        epoch_loss = 0.0
        epoch_correct = 0

        for label, pixels in data:
            # Your code here.
            pass

        avg_loss = epoch_loss / len(data)
        avg_acc = epoch_correct / len(data)
        print(f'Epoch {epoch + 1}: loss = {avg_loss:.4f}   accuracy = {avg_acc:.2%}')

    return loss_history, accuracy_history

In [ ]:
# Test it: train for a few epochs on our 200-example dataset.
# With only 200 examples and a randomly initialised network,
# do not expect dramatic results — but you should see the loss
# moving in the right direction.
net = make_network([784, 128, 64, 10])
losses, accs = train(net, data, epochs=3, learning_rate=0.01)

---

### Part 14: Testing with a Random Input

Before we move to the full dataset (that is Day 3), let us do a final sanity check: feed a random 784-value input through the trained network and look at what comes out.

In [ ]:
# Create a random input the right size.
random_input = [random.uniform(0, 1) for _ in range(784)]

# Run it through and look at the output.
layer_outputs = forward(net, random_input)
pred = predict(net, random_input)
output_probs = layer_outputs[-1][1]

print(f'Prediction: {pred}')
print(f'Confidence: {output_probs[pred]:.2%}')
print()
print('Full probability distribution:')
for digit, p in enumerate(output_probs):
    bar = '#' * int(p * 50)
    print(f'  {digit}: {bar:<50} {p:.4f}')

---

### Part 15: Assembling the Class

You now have every piece of the network working as a standalone function. Let us bring them all together into a single `NeuralNetwork` class.

Each function you wrote becomes a method. Where a function took `net` as its first argument, that becomes `self`. The internal helpers (random matrix, relu, softmax, weighted sum, forward layer) become private methods prefixed with an underscore.

In [ ]:
import random
import math

class NeuralNetwork:
    """A feedforward neural network for digit classification."""

    def __init__(self, layer_sizes):
        """
        Initialise the network.
        layer_sizes: e.g. [784, 128, 64, 10]
        """
        self.layer_sizes = layer_sizes
        self.weights = []
        self.biases = []
        for i in range(len(layer_sizes) - 1):
            self.weights.append(self._random_matrix(layer_sizes[i + 1], layer_sizes[i]))
            self.biases.append([0.0] * layer_sizes[i + 1])

    def __str__(self):
        """Return a readable summary of the network architecture."""
        # Your code here.
        pass

    def _random_matrix(self, rows, cols):
        """Return a matrix of random values between -0.1 and 0.1."""
        # Bring your random_matrix function here.
        pass

    def _relu(self, x):
        """Return x if positive, otherwise 0."""
        # Your code here.
        pass

    def _softmax(self, values):
        """Convert a list of values to a probability distribution."""
        # Your code here.
        pass

    def _weighted_sum(self, weights, inputs, bias):
        """Dot product of weights and inputs, plus bias."""
        # Your code here.
        pass

    def _forward_layer(self, inputs, layer_idx, activation):
        """
        Compute the output of one layer.
        Returns (pre_activation, post_activation).
        """
        # Your code here.
        pass

    def forward(self, inputs):
        """
        Pass inputs through all layers.
        Returns a list of (pre, post) tuples, one per non-input layer.
        """
        # Your code here.
        pass

    def predict(self, inputs):
        """Return the predicted digit for a given input."""
        # Your code here.
        pass

    def _cross_entropy_loss(self, output_probs, label):
        """Compute cross-entropy loss for one example."""
        # Your code here.
        pass

    def _backprop(self, inputs, label, layer_outputs, learning_rate):
        """
        Update weights using backpropagation.
        Bring across the provided backprop function,
        replacing net['weights'] with self.weights and so on.
        """
        # Your code here.
        pass

    def train(self, data, epochs=1, learning_rate=0.01):
        """
        Train on data. Print loss and accuracy per epoch.
        Returns (loss_history, accuracy_history).
        """
        # Your code here.
        pass

In [ ]:
# Final test for Day 2.
nn = NeuralNetwork([784, 128, 64, 10])
print(nn)

In [ ]:
# Quick training run to confirm the class works end-to-end.
losses, accs = nn.train(data, epochs=3, learning_rate=0.01)

In [ ]:
# Test the predict method directly on a few examples.
for label, pixels in data[:5]:
    pred = nn.predict(pixels)
    print(f'  True: {label}   Predicted: {pred}')

That is the end of Day 2. Your `NeuralNetwork` class is complete and working. In Day 3 we will load the full MNIST dataset, train for several epochs, and look carefully at what the network has learned.